# 🧪 Robot Arm PHM: Augmentation Benchmarking & Evaluation

This notebook evaluates the physical fidelity of synthetic robotic telemetry. We compare **TimeWarp** (Statistical) and **TimeGAN** (Generative) based on their ability to maintain distribution similarity and boundary continuity.

### 1️⃣ Cell 1: Environment Verification
Ensuring Python 3.10 and environment alignment.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate

# Assert Python 3.10
print(f"[CHECK] Python Version: {sys.version}")
assert sys.version_info.major == 3 and sys.version_info.minor == 10

try:
    from tsaug import TimeWarp
    from ydata_synthetic.synthesizers.timeseries import TimeSeriesSynthesizer
    print("[SUCCESS] Dependencies verified.")
except ImportError:
    print("[ERROR] Run 'pip install -r requirements.txt' in your Data_Degradation_env.")

from augmentation.expander import DataExpander
from utils.visualizer import Visualizer
print("[SUCCESS] Pipeline modules loaded.")

### 2️⃣ Cell 2: Data Loading & Initial Quality Check

In [ ]:
base_file = 'ready_for_ai.csv'
df = pd.read_csv(base_file)
print(f"[SUCCESS] Loaded Baseline: {df.shape[0]} rows")

time_col = [c for c in df.columns if 'time' in c.lower() or c.lower() == 't'][0]
delta_t = df[time_col].diff().mean()
print(f"Temporal Spacing Trace: Δt = {delta_t:.6f}s")

### 3️⃣ Cell 3: Comparative Augmentation Run
Executing $2\times$ expansion benchmarks.

In [ ]:
expander = DataExpander(base_file)
features = expander.features

print("[*] Running TimeWarp Expansion...")
df_warp = expander.expand(factor=2, method='warp')

print("\n[*] Running TimeGAN Expansion (20 Epochs for verification)...")
df_gan = expander.expand(factor=2, method='gan', epochs=20)

### 4️⃣ Cell 4: Mathematical Continuity & Distribution Evaluation
Calculating Euclidean Boundary Jump (Δ) and Maximum Mean Discrepancy (MMD Proxy).

In [ ]:
def calculate_metrics(df_expanded, df_real, features):
    orig_len = len(df_real)
    
    # 1. Boundary Euclidean Jump (Δ)
    prev_pt = df_expanded.iloc[orig_len - 1][features].values
    next_pt = df_expanded.iloc[orig_len][features].values
    delta = np.linalg.norm(prev_pt - next_pt)
    
    # 2. Simplified MMD (Distribution Similarity Score)
    # We compare the mean/std vectors of real vs synthetic data
    real_mu, synth_mu = df_real[features].mean().values, df_expanded.iloc[orig_len:][features].mean().values
    mmd_proxy = np.linalg.norm(real_mu - synth_mu)
    
    return delta, mmd_proxy

d_warp, m_warp = calculate_metrics(df_warp, df, features)
d_gan, m_gan = calculate_metrics(df_gan, df, features)

table = [
    ["Metric", "Target", "TimeWarp", "TimeGAN"],
    ["Boundary Jump (Δ)", "< 0.05", f"{d_warp:.5f}", f"{d_gan:.5f}"],
    ["MMD Proxy", "< 0.02", f"{m_warp:.5f}", f"{m_gan:.5f}"]
]
print(tabulate(table, headers="firstrow", tablefmt="fancy_grid"))

### 5️⃣ Cell 5: Automated Benchmarking Plots

In [ ]:
viz = Visualizer()
target = features[1] # Actual Current

print("Rendering TimeWarp Smoothing Analysis...")
viz.plot_stitching(df_warp, len(df), target)

print("Rendering TimeGAN Smoothing Analysis...")
viz.plot_stitching(df_gan, len(df), target)

# Display inline
from IPython.display import Image, display
display(Image(filename=f'plots/stage1_stitching_{target}.png'))